# Домашнє завдання №4 — Context Engineering

**Тема:** аналіз і генерація коротких текстів за допомогою OpenAI API.

**Модель:** `gpt-5.6-luna`.

**Підхід із лекції №4:** чіткі інструкції, відокремлення даних користувача від інструкцій, використання прикладів там, де це доцільно, обмеження формату відповіді та перевірка результатів.

У кожному завданні використано **Structured Outputs (сувору JSON-схему)** для забезпечення потрібної структури відповіді. Промпти визначають правила аналізу та генерації тексту, а локальні тести перевіряють правильність отриманих результатів.

Очікувані значення використовуються **лише для локальної перевірки** й не передаються моделі через API.

In [1]:
# Встановлюємо бібліотеку
%pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 49.9 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [3]:
# ЗАВДАННЯ 1 — Персона та тональність

import os
import json
from getpass import getpass
from openai import OpenAI


# 1. Перевіряємо змінну середовища
api_key = os.getenv("OPENAI_API_KEY")

# 2. Якщо ключа немає, шукаємо в Google Colab Secrets
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None

# 3. Якщо ключ не знайдено, просимо ввести вручну
if not api_key:
    api_key = getpass("OpenAI API key: ")

# 4. Зберігаємо ключ для поточної сесії
if not api_key or not api_key.strip():
    raise ValueError("OpenAI API key is required")

os.environ["OPENAI_API_KEY"] = api_key.strip()

# 5. Ініціалізуємо OpenAI
client = OpenAI()

MODEL = "gpt-5.6-luna"

texts_1 = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]
expected_1 = [
    {"person": "Оксана Дрозд", "tone": "позитивна"},
    {"person": "Андрій Король", "tone": "негативна"},
    {"person": "Роман Бондаренко", "tone": "нейтральна"},
    {"person": "Ілля Ткачук", "tone": "позитивна"},
    {"person": "Марко Савченко", "tone": "негативна"},
]

PROMPT_STUDENT_1 = """
Ти аналізуєш короткі повідомлення українською мовою.
Твоє завдання: знайти єдину згадану людину і визначити тональність саме щодо неї.

Правила:
1. person — дослівно перепиши ім'я та прізвище людини з тексту.
   Не включай посаду, звертання, пояснення або розділові знаки.
2. tone — тільки одне зі значень: позитивна, нейтральна, негативна.
   позитивна: текст хвалить людину або позитивно оцінює її дії/результат;
   негативна: текст критикує людину, її поведінку або негативний наслідок її дій;
   нейтральна: повідомлено факти про людину без оцінного судження.
3. Не визначай тональність окремих слів без урахування того, як вони стосуються людини.
4. Відповідь — лише один JSON-об'єкт із двома ключами person і tone.
   Не додавай коментарів, Markdown або інших полів.

Приклади формату (це демонстрації, а не тексти для оцінювання):
Текст: «Ганна Климчук чудово провела презентацію».
Відповідь: {"person":"Ганна Климчук","tone":"позитивна"}
Текст: «Павло Коваль повідомив дату зустрічі».
Відповідь: {"person":"Павло Коваль","tone":"нейтральна"}
Текст: «Леся Мороз не виконала обіцяне завдання».
Відповідь: {"person":"Леся Мороз","tone":"негативна"}
""".strip()

SCHEMA_1 = {
    "type": "json_schema",
    "json_schema": {
        "name": "person_tone",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "person": {"type": "string"},
                "tone": {"type": "string", "enum": ["позитивна", "нейтральна", "негативна"]},
            },
            "required": ["person", "tone"],
            "additionalProperties": False,
        },
    },
}

results_1 = []
for i, text in enumerate(texts_1, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_1},
            {"role": "user", "content": f"Проаналізуй повідомлення:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_1,
        reasoning_effort="none",  # для короткої класифікації без зайвих витрат на reasoning
    )
    raw = response.choices[0].message.content
    assert raw is not None, f"Порожня відповідь для прикладу {i}"
    parsed = json.loads(raw)  # Перевірка: відповідь — валідний JSON
    assert set(parsed) == {"person", "tone"}, f"Неправильні ключі: {parsed}"
    assert parsed["tone"] in {"позитивна", "нейтральна", "негативна"}
    results_1.append(parsed)
    print(f"{i}. {raw}")

passed_1 = sum(actual == expected for actual, expected in zip(results_1, expected_1))
print(f"Завдання 1: {passed_1}/{len(texts_1)} правильних відповідей")
assert passed_1 == len(texts_1), "Не всі відповіді збігаються з очікуваними"


1. {"person":"Оксана Дрозд","tone":"позитивна"}
2. {"person":"Андрій Король","tone":"негативна"}
3. {"person":"Роман Бондаренко","tone":"нейтральна"}
4. {"person":"Ілля Ткачук","tone":"позитивна"}
5. {"person":"Марко Савченко","tone":"негативна"}
Завдання 1: 5/5 правильних відповідей


In [10]:
# Перевіряємо промпт на інших реченнях без асертів і прикладів
test_texts_1 = [
    "Ірина Анатоліївна пречудово виконала композицію.",
    "Олег Бойко не впорався зі складним елементом на пілоні.",
    "Вчора Ганна Федосіївна провела лекцію з української мови для студентів-першокурсників.",
]

results_1 = []
for i, text in enumerate(test_texts_1, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_1},
            {"role": "user", "content": f"Проаналізуй повідомлення:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_1,
        reasoning_effort="none",  # для короткої класифікації без зайвих витрат на reasoning
    )
    raw = response.choices[0].message.content
    results_1.append(parsed)
    print(f"{i}. {raw}")

1. {"person":"Ірина Анатоліївна","tone":"позитивна"}
2. {"person":"Олег Бойко","tone":"негативна"}
3. {"person":"Ганна Федосіївна","tone":"нейтральна"}


### Завдання 2 — продукт і пріоритет
Використано **zero-shot** з правилами ухвалення рішення: домінантний вплив на роботу важливіший за окремі позитивні чи негативні слова. Текст `Atlas Pro` потрібно нормалізувати до `Atlas`. Ручний обхід проблеми не обов'язково робить фінансовий дефект низькопріоритетним.


In [7]:
# ЗАВДАННЯ 2 — Тріаж відгуків (продукт + пріоритет)
texts_2 = [
    "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
    "Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно.",
    "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
    "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
    "Atlas показує старі курси валют, поки що оновлюємо вручну.",
]
expected_2 = [
    {"product": "Atlas", "priority": "високий"},
    {"product": "Core", "priority": "середній"},
    {"product": "Desk", "priority": "низький"},
    {"product": "Core", "priority": "високий"},
    {"product": "Atlas", "priority": "високий"},
]

PROMPT_STUDENT_2 = """
Ти виконуєш тріаж коротких повідомлень про програмні продукти.
Для кожного повідомлення визнач ОДИН продукт і ОДИН пріоритет.

Продукт:
- Допустимі значення лише Atlas, Core, Desk.
- «Atlas Pro» — це варіант Atlas. Завжди повертай саме «Atlas».
- Не додавай версію продукту або пояснення.

Пріоритет визначай за найсуттєвішим впливом на роботу:
- високий: ключовий процес заблоковано, застосунок падає або дані критично
  неправильні (наприклад, застарілі фінансові курси); тимчасовий ручний
  обхід не усуває високого впливу.
- середній: поточна функція працює, але помітно погіршилась швидкість або зручність.
- низький: побажання, необов'язкова функція чи незначне покращення без збою.
Якщо є кілька сигналів, обери домінантний пріоритет за реальним впливом.

Повертай тільки валідний JSON-об'єкт із рівно двома ключами:
product і priority. Не додавай пояснень, Markdown чи додаткових полів.
""".strip()

SCHEMA_2 = {
    "type": "json_schema",
    "json_schema": {
        "name": "product_priority",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "product": {"type": "string", "enum": ["Atlas", "Core", "Desk"]},
                "priority": {"type": "string", "enum": ["високий", "середній", "низький"]},
            },
            "required": ["product", "priority"],
            "additionalProperties": False,
        },
    },
}

results_2 = []
for i, text in enumerate(texts_2, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_2},
            {"role": "user", "content": f"Класифікуй повідомлення:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_2,
        reasoning_effort="none",
    )
    raw = response.choices[0].message.content
    assert raw is not None, f"Порожня відповідь для прикладу {i}"
    parsed = json.loads(raw)
    assert set(parsed) == {"product", "priority"}, f"Неправильні ключі: {parsed}"
    assert parsed["product"] in {"Atlas", "Core", "Desk"}
    assert parsed["priority"] in {"високий", "середній", "низький"}
    results_2.append(parsed)
    print(f"{i}. {raw}")

passed_2 = sum(actual == expected for actual, expected in zip(results_2, expected_2))
print(f"Завдання 2: {passed_2}/{len(texts_2)} правильних відповідей")
assert passed_2 == len(texts_2), "Не всі відповіді збігаються з очікуваними"


1. {"product":"Atlas","priority":"високий"}
2. {"product":"Core","priority":"середній"}
3. {"product":"Desk","priority":"низький"}
4. {"product":"Core","priority":"високий"}
5. {"product":"Atlas","priority":"високий"}
Завдання 2: 5/5 правильних відповідей


In [8]:
# Зробимо перевірку промпта без асертів і без очікуваних відповідей
test_texts_2 = [
    "Desk перестав отримувати нові звернення клієнтів.",
    "У Core пошук став повільнішим, але результати правильні.",
    "Хотілося б додати темну тему в Atlas Pro."
]

results_2 = []
for i, text in enumerate(test_texts_2, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_2},
            {"role": "user", "content": f"Класифікуй повідомлення:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_2,
        reasoning_effort="none",
    )
    raw = response.choices[0].message.content
    print(f"{i}. {raw}")

1. {"product":"Desk","priority":"високий"}
2. {"product":"Core","priority":"середній"}
3. {"product":"Atlas","priority":"низький"}


### Завдання 3 — короткий заголовок і категорія
Категорія **«інцидент»** за правилами саме цього домашнього завдання включає також *планове технічне обслуговування*. **«Рекомендація»** включає прохання надати відгук. Заголовок перевіряється програмно: не більше восьми слів.


In [14]:
# ЗАВДАННЯ 3 — Генерація заголовка + категорія оголошення
texts_3 = [
    "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
    "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
    "Збираємо відгуки про новий модуль аналітики.",
    "На вихідних сервіс у режимі планового обслуговування.",
    "Рекомендуємо вмикати двофакторну автентифікацію.",
]
expected_categories_3 = [
    "реліз", "інцидент", "рекомендація", "інцидент", "рекомендація"
]

PROMPT_STUDENT_3 = """
Ти редагуєш короткі інформаційні оголошення українською.
Потрібно згенерувати лаконічний заголовок та класифікувати повідомлення.

Вимоги до заголовка (title):
- Не більше 8 слів; один рядок, українською мовою.
- Передай основний зміст тексту, без вигаданих подробиць і перебільшень.
- Не додавай службових префіксів на кшталт «Оголошення:».
- Версії, факти про несправність і рекомендації не підміняй іншими подіями.

Категорія (category) — тільки одне з трьох значень:
- реліз: вихід нової версії чи нової можливості;
- інцидент: несправність, порушення доступності АБО повідомлення про
  планове технічне обслуговування (згідно з категоріями цього завдання);
- рекомендація: порада користувачам, заклик виконати дію або надати відгук.

Поверни ЛИШЕ валідний JSON з рівно двома ключами: title та category.
Ніяких пояснень, Markdown, додаткових полів.
""".strip()

SCHEMA_3 = {
    "type": "json_schema",
    "json_schema": {
        "name": "title_category",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "category": {"type": "string", "enum": ["реліз", "інцидент", "рекомендація"]},
            },
            "required": ["title", "category"],
            "additionalProperties": False,
        },
    },
}

results_3 = []
for i, text in enumerate(texts_3, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_3},
            {"role": "user", "content": f"Сформуй заголовок і категорію:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_3,
        reasoning_effort="none",
    )
    raw = response.choices[0].message.content
    assert raw is not None, f"Порожня відповідь для прикладу {i}"
    parsed = json.loads(raw)
    assert set(parsed) == {"title", "category"}, f"Неправильні ключі: {parsed}"
    assert isinstance(parsed["title"], str) and parsed["title"].strip()
    assert parsed["category"] in {"реліз", "інцидент", "рекомендація"}
    word_count = len(parsed["title"].split())
    assert word_count <= 8, f"Заголовок має {word_count} слів: {parsed['title']}"
    results_3.append(parsed)
    print(f"{i}. {raw}")

passed_3 = sum(
    actual["category"] == expected
    for actual, expected in zip(results_3, expected_categories_3)
)
print(f"Завдання 3: {passed_3}/{len(texts_3)} правильних категорій; усі заголовки до 8 слів")
assert passed_3 == len(texts_3), "Не всі категорії збігаються з очікуваними"
print()
print(f"ЗАГАЛЬНИЙ РЕЗУЛЬТАТ: {passed_1 + passed_2 + passed_3}/15")


1. {"title":"Вийшла версія 2.0 з офлайн-режимом","category":"реліз"}
2. {"title":"Профіль не відкривається після оновлення","category":"інцидент"}
3. {"title":"Збираємо відгуки про модуль аналітики","category":"рекомендація"}
4. {"title":"Сервіс буде на плановому обслуговуванні","category":"інцидент"}
5. {"title":"Рекомендуємо ввімкнути двофакторну автентифікацію","category":"рекомендація"}
Завдання 3: 5/5 правильних категорій; усі заголовки до 8 слів

ЗАГАЛЬНИЙ РЕЗУЛЬТАТ: 15/15


In [13]:
# Перевіримо промпт на інших даних і виведемо кількість слів
test_texts_3 = [
    "Нова версія застосунку вже доступна за посиланням.",
    "Деякі існуючі користувачі не можуть залогінись із валідними кредами; можливо пов'язано з тим, коли саме вони створили профілі",
    "Будь ласка, натисніть на значок сповіщення.",
]

results_3 = []
for i, text in enumerate(test_texts_3, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT_STUDENT_3},
            {"role": "user", "content": f"Сформуй заголовок і категорію:\n<<<{text}>>>"},
        ],
        response_format=SCHEMA_3,
        reasoning_effort="none",
    )
    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    word_count = len(parsed["title"].split())
    results_3.append(parsed)
    print(f"{i}. {raw}  [слів: {word_count}]")

1. {"title":"Доступна нова версія застосунку","category":"реліз"}  [слів: 4]
2. {"title":"Деякі користувачі не можуть увійти","category":"інцидент"}  [слів: 5]
3. {"title":"Натисніть на значок сповіщення","category":"рекомендація"}  [слів: 4]


### Короткі висновки

- **Результати:** усі три завдання успішно пройшли автоматичні перевірки — **15/15 правильних відповідей**. Додатково я перевірила промпти завдань на нових текстах. Усі приклади модель обробила відповідно до очікувань.
- **Промптинг:** у першому завданні використано few-shot із трьома прикладами, у другому та третьому — zero-shot із чіткими правилами класифікації. Обидва підходи спрацювали на тестових даних.
- **Структурований вивід:** JSON Schema допомогла забезпечити потрібний формат відповідей, а перевірки Python — перевірити допустимі значення, правильність класифікації та обмеження довжини заголовків.
- **Якість відповідей:** правильний формат і проходження тестів ще не гарантують повноти змісту. Наприклад, у третьому завданні модель сформувала коректний заголовок, але пропустила інформацію про новий дизайн застосунку.
- **Стабільність:** результати позитивні, зокрема на нових прикладах. Проте одного запуску недостатньо, щоб зробити висновок про стабільність. Для цього потрібно більше тестових даних і повторні запуски.

**Загальний висновок:** чіткі інструкції, обмеження формату та правила класифікації допомогли отримати правильні результати. Водночас важливо перевіряти не лише структуру відповіді, а й її зміст, повноту та поведінку моделі на нових даних.